In [ ]:
!rm -rf '/content/DIS_Hughen'
!git clone https://github.com/NU-Academics/DIS_Hughen.git

In [1]:
import numpy as np
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import  accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset, DataLoader
from xgboost import XGBClassifier

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [8]:
df = pd.read_csv("undersampled_CIC2019_dataset.csv")
# df = pd.read_csv("/content/DIS_Hughen/undersampled_CIC2019_dataset.csv")

In [41]:
df.shape

(4685611, 90)

In [42]:
attack_set = df[df["label"] != "BENIGN"]
attack_sample = (attack_set.groupby("label", group_keys=False).sample(n=2000, random_state=42, replace=True).reset_index(drop=True))

In [43]:
attack_sample.shape

(36000, 90)

In [44]:
print(attack_sample["label"].value_counts())

label
DrDoS_DNS        2000
DrDoS_LDAP       2000
DrDoS_MSSQL      2000
DrDoS_NTP        2000
DrDoS_NetBIOS    2000
DrDoS_SNMP       2000
DrDoS_SSDP       2000
DrDoS_UDP        2000
LDAP             2000
MSSQL            2000
NetBIOS          2000
Portmap          2000
Syn              2000
TFTP             2000
UDP              2000
UDP-lag          2000
UDPLag           2000
WebDDoS          2000
Name: count, dtype: int64


In [45]:
benign_set = df[df["label"] == "BENIGN"]
benign_sample = benign_set.sample(n=attack_sample.shape[0], replace=True, random_state=42)

In [46]:
benign_sample.shape

(36000, 90)

In [47]:
print(benign_sample["label"].value_counts())

label
BENIGN    36000
Name: count, dtype: int64


In [48]:
df_sample =pd.concat([benign_sample, attack_sample]).reset_index(drop=True)

In [49]:
df_sample.shape

(72000, 90)

In [50]:
print(df_sample["label"].value_counts())

label
BENIGN           36000
DrDoS_DNS         2000
DrDoS_LDAP        2000
DrDoS_MSSQL       2000
DrDoS_NTP         2000
DrDoS_NetBIOS     2000
DrDoS_SNMP        2000
DrDoS_SSDP        2000
DrDoS_UDP         2000
LDAP              2000
MSSQL             2000
NetBIOS           2000
Portmap           2000
Syn               2000
TFTP              2000
UDP               2000
UDP-lag           2000
UDPLag            2000
WebDDoS           2000
Name: count, dtype: int64


In [51]:
df_sample["label"] = df_sample["label"].apply(lambda x: 0 if x == "BENIGN" else 1)
X = df_sample.drop(columns=["label"]).select_dtypes(include=[np.number])
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0, inplace=True)
y = df_sample["label"]

In [52]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [53]:
class CICDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = CICDataset(X_train, y_train)
train_loader = DataLoader(train_dataset,
                          batch_size=64, #8192
                          shuffle=True,
                          pin_memory=True,
                          num_workers=0) #4

class DNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(128, 64)
        self.output = nn.Linear(64, num_classes)
    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        features = F.relu(self.fc3(x))
        logits = self.output(features)
        return logits, features
        
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

In [54]:
X_train2, X_val, y_train2, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.1,        # 10% validation
    stratify=y_train,     # important for imbalance
    random_state=42
)

model = DNN(input_dim=X_train.shape[1], num_classes=len(np.unique(y_train))).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

best_val_loss = float("inf")
patience = 3
counter = 0

for epoch in range(30):  # max epochs
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits, _ = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Validation
    model.eval()
    with torch.no_grad():
        X_val_tensor = torch.tensor(X_val.values,
                                    dtype=torch.float32).to(device)
        y_val_tensor = torch.tensor(y_val.values,
                                    dtype=torch.long).to(device)

        val_logits, _ = model(X_val_tensor)
        val_loss = criterion(val_logits, y_val_tensor)

    print(f"Epoch {epoch+1} | Train Loss: {total_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_dnn.pt")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered.")
            break

Epoch 1 | Train Loss: 88.6944 | Val Loss: 0.0198
Epoch 2 | Train Loss: 32.5263 | Val Loss: 0.0110
Epoch 3 | Train Loss: 25.3227 | Val Loss: 0.0113
Epoch 4 | Train Loss: 21.7290 | Val Loss: 0.0163
Epoch 5 | Train Loss: 20.0040 | Val Loss: 0.0127
Early stopping triggered.


In [55]:
def extract_features(model, X, batch_size=8192):
    model.eval()
    dataset = torch.tensor(X.values, dtype=torch.float32)
    loader = DataLoader(dataset, batch_size=batch_size)

    features_list = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            _, features = model(batch)
            features_list.append(features.cpu().numpy())

    return np.vstack(features_list)

Z_train = extract_features(model, X_train)
Z_test = extract_features(model, X_test)

In [56]:
X_train_hybrid = np.hstack([X_train.values, Z_train])
X_test_hybrid = np.hstack([X_test.values, Z_test])

In [57]:
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method='hist',
#    device='cuda',   # remove if CPU
    objective='binary:logistic',
    eval_metric='logloss'
)

param_dist = {
    "max_depth": [6, 7, 8, 9],
    "learning_rate": [0.03, 0.05, 0.07],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.2]
}

search = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1_macro",
    cv=3,
    verbose=2
)

search.fit(X_train_hybrid, y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END colsample_bytree=0.8, gamma=0.2, learning_rate=0.07, max_depth=6, min_child_weight=1, subsample=0.8; total time=   1.9s
[CV] END colsample_bytree=0.8, gamma=0.2, learning_rate=0.07, max_depth=6, min_child_weight=1, subsample=0.8; total time=   1.7s
[CV] END colsample_bytree=0.8, gamma=0.2, learning_rate=0.07, max_depth=6, min_child_weight=1, subsample=0.8; total time=   1.6s
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.03, max_depth=8, min_child_weight=5, subsample=0.8; total time=   1.6s
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.03, max_depth=8, min_child_weight=5, subsample=0.8; total time=   1.7s
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.03, max_depth=8, min_child_weight=5, subsample=0.8; total time=   1.7s
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.05, max_depth=6, min_child_weight=3, subsample=0.9; total time=   1.7s
[CV] END colsample_bytree=0.8, gamma

,estimator,"XGBClassifier...ree=None, ...)"
,param_distributions,"{'colsample_bytree': [0.7, 0.8, ...], 'gamma': [0, 0.1, ...], 'learning_rate': [0.03, 0.05, ...], 'max_depth': [6, 7, ...], ...}"
,n_iter,20
,scoring,'f1_macro'
,n_jobs,None
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [58]:
y_pred = search.predict(X_test_hybrid)
y_prob = search.predict_proba(X_test_hybrid)[:, 1]
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Weighted F1:", f1_score(y_test, y_pred, average='weighted'))
print("Macro F1:", f1_score(y_test, y_pred, average='macro'))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.9997916666666666
Weighted F1: 0.9997916666656619
Macro F1: 0.999791666665662
ROC-AUC: 0.9999998263888888


In [59]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7200
           1       1.00      1.00      1.00      7200

    accuracy                           1.00     14400
   macro avg       1.00      1.00      1.00     14400
weighted avg       1.00      1.00      1.00     14400

